In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The [Zebra Puzzle](https://en.wikipedia.org/wiki/Zebra_Puzzle)

The following puzzle appeared in the magazine *Life International* on the 17th of December in the year 1962:
* There are five houses.
* The Englishman lives in the red house.
* The Spaniard owns the dog.
* Coffee is drunk in the green house.
* The Ukrainian drinks tea.
* The green house is immediately to the right of the ivory house.
* The Old Gold smoker owns snails.
* Kools are smoked in the yellow house.
* Milk is drunk in the middle house.
* The Norwegian lives in the first house.
* The man who smokes Chesterfields lives in the house next to the man with the fox.
* Kools are smoked in the house next to the house where the horse is kept.
* The Lucky Strike smoker drinks orange juice.
* The Japanese smokes Parliaments.
* The Norwegian lives next to the blue house.

Furthermore, each of the five houses is painted in a different colour, their inhabitants are of different nationalities, own different pets, drink different beverages, and smoke different brands of cigarettes.

Our task is to answers the following questions: 
1. *Who drinks water?*
2. *Who owns the zebra?*

## Choosing the Appropriate Variables

In order to solve this problem we use the following propositional variables, where the counter $i$ ranges from $1$ to $5$.  

* $\texttt{English}\langle i \rangle$ expresses the fact
  that the Englishman lives in house number $i$.
  
  The remaining nationalities are coded using the variables
  $\texttt{Spanish}\langle i \rangle$, $\texttt{Ukrainian}\langle i \rangle$,
  $\texttt{Norwegian}\langle i \rangle$, and $\texttt{Japanese}\langle i \rangle$.
* $\mathtt{Red}\langle i \rangle$ expresses that house number $i$ is red.
  
  We use the variables $\texttt{Green}\langle i \rangle$, $\texttt{Ivory}\langle i \rangle$,
  $\texttt{Yellow}\langle i \rangle$, and $\texttt{Blue}\langle i \rangle$ to encode the remaining
  colors.
* $\texttt{OldGold}\langle i \rangle$ expresses that the inhabitant of house number $i$ smokes cigarettes of the brand "Old Gold".
  
  We use the variables $\texttt{Kools}\langle i \rangle$,
  $\texttt{Chesterfields}\langle i \rangle$, $\texttt{LuckyStrike}\langle i \rangle$, and
  $\texttt{Parliaments}\langle i \rangle$ to encode the cigarette brands.
* $\texttt{Dog}\langle i \rangle$ expresses the fact that the inhabitant of house number
  $i$ keeps a dog as his pet.

  We use the variables $\texttt{Snails}\langle i \rangle$, $\texttt{Fox}\langle i \rangle$,
  $\texttt{Horse}\langle i \rangle$, and $\texttt{Zebra}\langle i \rangle$ to encode the
  remaining pets.
* $\texttt{Coffee}\langle i \rangle$ expresses the fact the the inhabitant of house number $i$
  drinks coffee.

  We use the variables $\texttt{Milk}\langle i \rangle$,
  $\texttt{OrangeJuice}\langle i \rangle$, $\texttt{Tea}\langle i \rangle$, 
  and $\texttt{Water}\langle i \rangle$ to encode the remaining drinks.

We are using the angular brackets "$\langle$" and "$\rangle$" as part of the variable names 
because our parser for propositional logic accepts these symbols as part of variable names.  The parser would be confused if we would use parentheses.

## Importing the Necessary Notebooks

Our goal is to solve this puzzle by first coding it as a solvability problem of propositional logic and then to solve the resulting set of clauses using the algorithm of Davis and Putnam.  We import the notebook `07-Davis-Putnam-JW.ipynb`.  This notebook imports the notebook `04-CNF.ipynb`, which implements the function `normalize` that takes a formula and transforms it into a set of clauses.  The notebook `04-CNF.ipynb` in turn imports the parser for propositional logic.  Hence, we have access to all functions defined in these three notebooks.

In [2]:
importNotebook "07-Davis-Putnam-JW.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb, 07-Davis-Putnam-JW.ipynb

Using the parser and the function `normalize` we can implement the function `parseKNF`.  It takes one argument:
- `s` is a string representing a propositional formula.

It returns a set of clauses that is equivalent to the formula represented by `s`.

In [3]:
let parseKNF (s: string) : CNF =
    normalize (parse s)

## Auxiliary Functions

In order to succinctly express the constraints that all houses have different colours, the inhabitants have different nationalities etc., it is convenient to implement a function `atMostOne`.  It takes one argument:
- `V` is a set of propositional variables.

It returns a set of clauses that is true if and only if at most one of the variables from $V$ is true.

In [4]:
let atMostOne (V: Set<string>) : CNF =
    set [ for p in V do
            for q in V do
                if p <> q then
                    set [ Neg p; Neg q ] ]

The function `var` takes two arguments:
- `f` is the name of a property, e.g. `"Japanese"`,
- `i` is the number of a house, i.e. $i \in\{1,2,3,4,5\}$.

It returns the string $f\langle i \rangle$, e.g. the call `var "Japanese" 2` returns the following string:
```
Japanese<2>
```

In [5]:
let var (f: string) (i: int) : string =
    sprintf "%s<%d>" f i

In [6]:
var "Japanese" 2

"Japanese<2>"

The function `somewhere` takes one argument:
- `x` is the name of a property.

It returns a clause that specifies that the person with property $x$ has to live in one of the houses from $1$ to $5$.

In [7]:
let somewhere (x: string) : Clause =
    set [ for i in 1 .. 5 -> Pos (var x i) ]

In [8]:
somewhere "Japanese"

set [Pos "Japanese<1>"; Pos "Japanese<2>"; Pos "Japanese<3>"; Pos "Japanese<4>"; Pos "Japanese<5>"]

The function `atMostOneAt` takes two arguments:
- `S` is a set of mutually exclusive properties,
- `i` is the number of a house.

It returns a set of clauses that specifies that the person living in house number $i$ has at most one of the properties from the set $S$.  For example, if 
$S = \{\texttt{"Japanese"}, \texttt{"English"}, \texttt{"Spanish"}, \texttt{"Norwegian"}, \texttt{"Ukrainian"}\}$, 
then $\texttt{atMostOneAt}(S, 3)$ specifies that the inhabitant of house number 3 has at most one of the nationalities from the set $S$.

In [9]:
let atMostOneAt (S: Set<string>) (i: int) : CNF =
    atMostOne (Set.map (fun x -> var x i) S)

In [10]:
atMostOneAt (set [ "A"; "B"; "C" ]) 1

set [set [Neg "A<1>"; Neg "B<1>"]; set [Neg "A<1>"; Neg "C<1>"]; set [Neg "B<1>"; Neg "C<1>"]]

The function `onePerHouse` takes one argument:
- `S` is a set of five mutually exclusive properties, e.g. the set of nationalities.

It returns a set of clauses that expresses that every property from $S$ belongs to the inhabitant of some house and that no inhabitant has two of the properties from $S$.  For example, the call
$$\texttt{onePerHouse}(\{\texttt{"Japanese"},
       \texttt{"English"}, 
       \texttt{"Spanish"}, \texttt{"Norwegian"}, 
       \texttt{"Ukrainian"}\})
$$
creates a set of clauses that expresses that there has to be a house where the Japanese lives, a house where the Englishman lives, a house where the Spaniard lives, a house where the Norwegian lives, and a house
where the Ukrainian lives.  Furthermore, the set of clauses contains clauses that express that these five persons live in **different** houses.

In [11]:
let onePerHouse (S: Set<string>) : CNF =
    set [ for x in S -> somewhere x ] + Set.unionMany [ for i in 1 .. 5 -> atMostOneAt S i ]

In [12]:
onePerHouse (set [ "A"; "B" ])

set [set [Pos "A<1>"; Pos "A<2>"; Pos "A<3>"; Pos "A<4>"; Pos "A<5>"]; set [Pos "B<1>"; Pos "B<2>"; Pos "B<3>"; Pos "B<4>"; Pos "B<5>"]; set [Neg "A<1>"; Neg "B<1>"]; set [Neg "A<2>"; Neg "B<2>"]; set [Neg "A<3>"; Neg "B<3>"]; set [Neg "A<4>"; Neg "B<4>"]; set [Neg "A<5>"; Neg "B<5>"]]

The function `sameHouse` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that if the inhabitant of house number $i$ has the property $a$, then he also has the property $b$ and vice versa.  For example, $\texttt{sameHouse}(\texttt{"Japanese"}, \texttt{"Dog"})$ specifies that the Japanese guy keeps a dog.

In [13]:
let sameHouse (a: string) (b: string) : CNF =
    Set.unionMany [ for i in 1 .. 5 -> parseKNF (sprintf "%s ↔ %s" (var a i) (var b i)) ]

In [14]:
sameHouse "Red" "Tea"

set [set [Pos "Red<1>"; Neg "Tea<1>"]; set [Pos "Red<2>"; Neg "Tea<2>"]; set [Pos "Red<3>"; Neg "Tea<3>"]; set [Pos "Red<4>"; Neg "Tea<4>"]; set [Pos "Red<5>"; Neg "Tea<5>"]; set [Pos "Tea<1>"; Neg "Red<1>"]; set [Pos "Tea<2>"; Neg "Red<2>"]; set [Pos "Tea<3>"; Neg "Red<3>"]; set [Pos "Tea<4>"; Neg "Red<4>"]; set [Pos "Tea<5>"; Neg "Red<5>"]]

The function `nextTo` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that the inhabitants with properties $a$ and $b$ are direct neighbours.  For example, $\texttt{nextTo}(\texttt{"Japanese"}, \texttt{"Dog"})$ specifies that the Japanese guy lives next to the guy who keeps a dog.  The houses $1$ and $5$ have only one neighbour, the houses $2$, $3$, and $4$ have two neighbours.

In [15]:
let nextTo (a: string) (b: string) : CNF =
    parseKNF (sprintf "%s → %s" (var a 1) (var b 2))
    + Set.unionMany [ for i in 2 .. 4 ->
                        parseKNF (sprintf "%s → %s ∨ %s" (var a i) (var b (i-1)) (var b (i+1))) ]
    + parseKNF (sprintf "%s → %s" (var a 5) (var b 4))

In [16]:
nextTo "A" "B"

set [set [Pos "B<1>"; Pos "B<3>"; Neg "A<2>"]; set [Pos "B<2>"; Pos "B<4>"; Neg "A<3>"]; set [Pos "B<2>"; Neg "A<1>"]; set [Pos "B<3>"; Pos "B<5>"; Neg "A<4>"]; set [Pos "B<4>"; Neg "A<5>"]]

The function `leftTo` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that the inhabitant with property $a$ lives in the house immediately to the left of the house of the inhabitant who has property $b$.  For example, $\texttt{leftTo}(\texttt{"Japanese"}, \texttt{"Dog"})$ specifies that the Japanese guy lives in the house to the left of the house where there is a dog.

In [17]:
let leftTo (a: string) (b: string) : CNF =
    Set.unionMany [ for i in 1 .. 4 -> parseKNF (sprintf "%s ↔ %s" (var a i) (var b (i+1))) ]
    + parseKNF (sprintf "¬%s" (var a 5))

In [18]:
leftTo "A" "B"

set [set [Pos "A<1>"; Neg "B<2>"]; set [Pos "A<2>"; Neg "B<3>"]; set [Pos "A<3>"; Neg "B<4>"]; set [Pos "A<4>"; Neg "B<5>"]; set [Pos "B<2>"; Neg "A<1>"]; set [Pos "B<3>"; Neg "A<2>"]; set [Pos "B<4>"; Neg "A<3>"]; set [Pos "B<5>"; Neg "A<4>"]; set [Neg "A<5>"]]

In [19]:
let Nations = set [ "English"; "Spanish"; "Ukrainian"; "Norwegian"; "Japanese" ]
let Drinks  = set [ "Coffee"; "Tea"; "Milk"; "OrangeJuice"; "Water" ]
let Pets    = set [ "Dog"; "Snails"; "Horse"; "Fox"; "Zebra" ]
let Brands  = set [ "LuckyStrike"; "Parliaments"; "Kools"; "Chesterfields"; "OldGold" ]
let Colours = set [ "Red"; "Green"; "Ivory"; "Yellow"; "Blue" ]

The function `allClauses` takes no arguments.  It returns a set of clauses describing the problem.

In [20]:
let allClauses () : CNF =
    // Every house has exactly one inhabitant.  This inhabitant has exactly one
    // nationality, one pet, smokes one brand of cigarettes, and has one type
    // of drink.  Furthermore, every house has exactly one color.
    let mutable Clauses = onePerHouse Nations
    Clauses <- Clauses + onePerHouse Drinks
    Clauses <- Clauses + onePerHouse Pets
    Clauses <- Clauses + onePerHouse Brands
    Clauses <- Clauses + onePerHouse Colours
    // The Englishman lives in the red house.
    Clauses <- Clauses + sameHouse "English" "Red"
    // The Spaniard owns the dog.
    Clauses <- Clauses + sameHouse "Spanish" "Dog"
    // Coffee is drunk in the green house.
    Clauses <- Clauses + sameHouse "Coffee" "Green"
    // The Ukrainian drinks tea.
    Clauses <- Clauses + sameHouse "Ukrainian" "Tea"
    // The green house is immediately to the right of the ivory house.
    Clauses <- Clauses + leftTo "Ivory" "Green"
    // The Old Gold smoker owns snails.
    Clauses <- Clauses + sameHouse "OldGold" "Snails"
    // Kools are smoked in the yellow house.
    Clauses <- Clauses + sameHouse "Kools" "Yellow"
    // Milk is drunk in the middle house.
    Clauses <- Clauses + parseKNF "Milk<3>"
    // The Norwegian lives in the first house.
    Clauses <- Clauses + parseKNF "Norwegian<1>"
    // The man who smokes Chesterfields lives in the house next 
    // to the man with the fox.
    Clauses <- Clauses + nextTo "Chesterfields" "Fox"
    // Kools are smoked in the house next to the house where the horse is kept.
    Clauses <- Clauses + nextTo "Kools" "Horse"
    // The Lucky Strike smoker drinks orange juice.
    Clauses <- Clauses + sameHouse "LuckyStrike" "OrangeJuice"
    // The Japanese smokes Parliaments.
    Clauses <- Clauses + sameHouse "Japanese" "Parliaments"
    // The Norwegian lives next to the blue house.
    Clauses <- Clauses + nextTo "Norwegian" "Blue"
    Clauses

In [21]:
let Clauses = allClauses ()
printfn "%s" (cnfToString Clauses)

{{Blue<1>, Blue<2>, Blue<3>, Blue<4>, Blue<5>}, {Blue<1>, Blue<3>, ¬Norwegian<2>}, {Blue<2>, Blue<4>, ¬Norwegian<3>}, {Blue<2>, ¬Norwegian<1>}, {Blue<3>, Blue<5>, ¬Norwegian<4>}, {Blue<4>, ¬Norwegian<5>}, {Chesterfields<1>, Chesterfields<2>, Chesterfields<3>, Chesterfields<4>, Chesterfields<5>}, {Coffee<1>, Coffee<2>, Coffee<3>, Coffee<4>, Coffee<5>}, {Coffee<1>, ¬Green<1>}, {Coffee<2>, ¬Green<2>}, {Coffee<3>, ¬Green<3>}, {Coffee<4>, ¬Green<4>}, {Coffee<5>, ¬Green<5>}, {Dog<1>, Dog<2>, Dog<3>, Dog<4>, Dog<5>}, {Dog<1>, ¬Spanish<1>}, {Dog<2>, ¬Spanish<2>}, {Dog<3>, ¬Spanish<3>}, {Dog<4>, ¬Spanish<4>}, {Dog<5>, ¬Spanish<5>}, {English<1>, English<2>, English<3>, English<4>, English<5>}, {English<1>, ¬Red<1>}, {English<2>, ¬Red<2>}, {English<3>, ¬Red<3>}, {English<4>, ¬Red<4>}, {English<5>, ¬Red<5>}, {Fox<1>, Fox<2>, Fox<3>, Fox<4>, Fox<5>}, {Fox<1>, Fox<3>, ¬Chesterfields<2>}, {Fox<2>, Fox<4>, ¬Chesterfields<3>}, {Fox<2>, ¬Chesterfields<1>}, {Fox<3>, Fox<5>, ¬Chesterfields<4>}, {Fox<4>, ¬

In [22]:
Clauses.Count

381

Solving the problem takes only a fraction of a second.

In [23]:
#!time
let Solution = solve Clauses

Wall time: 99.3696ms

## Functions to PrettyPrint the Solution

The function `extractAssignment` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the puzzle.

It returns a map that assigns to every property the number of the house it belongs to.  For example, if `Solution` contains the unit clause $\{\texttt{Dog<3>}\}$, then the map assigns the number $3$ to `"Dog"`.  Only the positive literals are relevant.  The functions `arb` and `complement` are defined in the notebook `07-Davis-Putnam-JW.ipynb`.

**Implementation:** For a variable of the form `name<i>`, `IndexOf '<'` returns the position of the character `<`.  The name is the part before this position, the house number is the part between `<` and the final `>`.

In [24]:
let extractAssignment (Solution: CNF) : Map<string, int> =
    Map [ for Unit in Solution do
            match arb Unit with
            | Pos v -> let k = v.IndexOf '<'
                       (v.Substring(0, k), int (v.Substring(k + 1, v.Length - k - 2)))
            | Neg _ -> () ]

In [25]:
extractAssignment Solution

map [("Blue", 2); ("Chesterfields", 2); ("Coffee", 5); ("Dog", 4); ("English", 3); ("Fox", 1); ("Green", 5); ("Horse", 2); ("Ivory", 4); ("Japanese", 5); ("Kools", 1); ("LuckyStrike", 4); ("Milk", 3); ("Norwegian", 1); ("OldGold", 3); ("OrangeJuice", 4); ("Parliaments", 5); ("Red", 3); ("Snails", 3); ("Spanish", 4); ("Tea", 2); ("Ukrainian", 2); ("Water", 1); ("Yellow", 1); ("Zebra", 5)]

The function `showHTML` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the puzzle.

It returns an HTML table that shows the solution.  The table has one row for every house.  The function `HTML` is provided by the notebook kernel.  It turns a string containing HTML into HTML content that is displayed by the notebook.

In [26]:
let showHTML (Solution: CNF) =
    let Assignment = extractAssignment Solution
    let sb = System.Text.StringBuilder()
    sb.Append("<table style=\"border:2px solid blue\">\n<tr>") |> ignore
    for name in [ "House"; "Nationality"; "Drink"; "Animal"; "Brand"; "Colour" ] do
        sb.Append("<th style=\"color:gold; background-color:blue\">" + name + "</th>") |> ignore
    sb.Append("</tr>\n") |> ignore
    for house in 1 .. 5 do
        sb.Append("<tr><td style=\"border:1px solid green\">" + string house + "</td>") |> ignore
        for Class in [ Nations; Drinks; Pets; Brands; Colours ] do
            for x in Class do
                if Assignment[x] = house then
                    sb.Append("<td style=\"border:1px solid green\">" + x + "</td>") |> ignore
        sb.Append("</tr>\n") |> ignore
    sb.Append("</table>") |> ignore
    HTML(sb.ToString())

In [27]:
showHTML Solution

House,Nationality,Drink,Animal,Brand,Colour
1,Norwegian,Water,Fox,Kools,Yellow
2,Ukrainian,Tea,Horse,Chesterfields,Blue
3,English,Milk,Snails,OldGold,Red
4,Spanish,OrangeJuice,Dog,LuckyStrike,Ivory
5,Japanese,Coffee,Zebra,Parliaments,Green


## Checking the Uniqueness of the Solution

The function `negateSolution` takes one argument:
- `UnitClauses` is a set of unit clauses.

It returns a clause that is the logical negation of the conjunction of these unit clauses.  This clause contains the complements of all literals in `UnitClauses`.

In [28]:
let negateSolution (UnitClauses: CNF) : Clause =
    Set.map (fun Unit -> complement (arb Unit)) UnitClauses

In [29]:
negateSolution (set [ set [ Pos "a" ]; set [ Neg "b" ] ])

set [Pos "b"; Neg "a"]

The function `checkUniqueness` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of `Clauses`,
- `Clauses` is a set of clauses.

It checks whether `Solution` is the only solution of `Clauses`.  To this end, the negation of `Solution` is added to `Clauses`.  If the resulting set of clauses is unsatisfiable, the solution is unique.

In [30]:
let checkUniqueness (Solution: CNF) (Clauses: CNF) : unit =
    let negation    = negateSolution Solution
    let alternative = solve (Clauses.Add negation)
    if alternative = set [ Set.empty ] then
        printfn "The solution is unique!"
    else
        printfn "ERROR: The solution is not unique!"

In [31]:
#!time
checkUniqueness Solution Clauses

The solution is unique!

Wall time: 160.1487ms